<a href="https://colab.research.google.com/github/Vronska-Anhelina/University-app-/blob/main/teachers_and_rooms_conflicts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

import zipfile

zip_path = "/content/drive/MyDrive/university_schema.zip"
extract_path = "/content/university_schema"
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)
path = extract_path
users = pd.read_csv(f"{path}/users.csv")
teachers = pd.read_csv(f"{path}/teachers.csv")
groups = pd.read_csv(f"{path}/groups.csv")
rooms = pd.read_csv(f"{path}/rooms.csv")
time_slot = pd.read_csv(f"{path}/time_slot.csv")
courses = pd.read_csv(f"{path}/courses.csv")
course_offerings = pd.read_csv(f"{path}/course_offerings.csv")
schedule_entries = pd.read_csv(f"{path}/schedule_entries.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = schedule_entries.merge(course_offerings, on="course_offering_id")
df = df.merge(rooms, left_on="room_id",right_on = "rooms_id")
df = df.merge(time_slot, on="time_slot_id")
df = df.merge(teachers, on="teacher_id")
df = df.merge(groups, on="group_id")
df = df.merge(courses, on="course_id")
df = df.merge(users, left_on="teacher_id", right_on = "id")

In [ ]:
df.head()

,course_offering_id,room_id,time_slot_id,day_of_week,course_id,group_id,teacher_id,semester_id,c,rooms_id,...,cathedra,name_y,specialty,year_of_study,name,id,full_name,role,email,phone_number
0,1,74,2,3,19,1,50,1,4,74,...,Physics,IPZ-1A,IPZ,1,Computer Graphics,50,пан Андрій Гайворонський,teacher,lutsenkodemyd@example.org,042 213-11-78
1,1,68,6,6,19,1,50,1,4,68,...,Physics,IPZ-1A,IPZ,1,Computer Graphics,50,пан Андрій Гайворонський,teacher,lutsenkodemyd@example.org,042 213-11-78
2,2,66,4,1,7,1,11,1,2,66,...,Computer Science,IPZ-1A,IPZ,1,Operating Systems,11,Орися Піддубна,teacher,svystunteodor@example.net,+380 02 935-37-02
3,2,49,3,4,7,1,11,1,2,49,...,Computer Science,IPZ-1A,IPZ,1,Operating Systems,11,Орися Піддубна,teacher,svystunteodor@example.net,+380 02 935-37-02
4,3,37,3,2,15,1,50,1,3,37,...,Physics,IPZ-1A,IPZ,1,Information Security,50,пан Андрій Гайворонський,teacher,lutsenkodemyd@example.org,042 213-11-78


In [ ]:
print(df.columns)

Index(['course_offering_id', 'room_id', 'time_slot_id', 'day_of_week',
       'course_id', 'group_id', 'teacher_id', 'semester_id', 'c', 'rooms_id',
       'building', 'number_x', 'type', 'number_y', 'time_start', 'time_end',
       'name_x', 'cathedra', 'name_y', 'specialty', 'year_of_study', 'name',
       'id', 'full_name', 'role', 'email', 'phone_number'],
      dtype='object')


In [ ]:
df = df[["full_name", "day_of_week", "time_slot_id", "semester_id", "rooms_id", "group_id","type"]]
df.head()

,full_name,day_of_week,time_slot_id,semester_id,rooms_id,group_id,type
0,пан Андрій Гайворонський,3,2,1,74,1,lecture_hall
1,пан Андрій Гайворонський,6,6,1,68,1,lecture_hall
2,Орися Піддубна,1,4,1,66,1,lecture_hall
3,Орися Піддубна,4,3,1,49,1,seminar
4,пан Андрій Гайворонський,2,3,1,37,1,lab


# Один викладач в двох різних аудиторіях

In [ ]:
teacher_conflicts = df[df.duplicated(subset = ["full_name","day_of_week","time_slot_id","semester_id","group_id"],
                                  keep = False)]

print(teacher_conflicts)

            full_name  day_of_week  time_slot_id  semester_id  rooms_id  \
1607  Пріска Піддубна            3             3            4         6   
1613  Пріска Піддубна            3             3            4        56   
2107     Віра Хоменко            2             3            5         3   
2108     Віра Хоменко            2             3            5        76   
2744   Василина Таран            1             6            6        29   
2754   Василина Таран            1             6            6        75   

      group_id          type  
1607        22  lecture_hall  
1613        22       seminar  
2107        23  lecture_hall  
2108        23       seminar  
2744        37  lecture_hall  
2754        37           lab  


Один викладач не може одночасно проводити заняття в двох різних аудиторіях для різних груп в один і той самий день та час в межах одного семестру. Тому перевіряється чи не дублюється викладач за параметрами : день + час + семестр + група.

* Пріска Піддубна - Середа, третя пара, 4 семестр, , але дві різні аудиторії 6 і 56.
* Віра Хоменко - Вівторок, третя пара ,5 семестр, аудиторії 3 і 76а
* Василина Таран - Понеділок,шоста пара,6 семестр, аудиторії 29 і 75.

In [ ]:
df = df.drop_duplicates(subset = ["full_name","day_of_week","time_slot_id","semester_id","group_id"]
                        ,keep="first")

In [ ]:
teacher_conflict_after = df[df.duplicated( #Перевіримо чи є ще дублікати після видалення
    subset=["full_name", "day_of_week", "time_slot_id", "semester_id","group_id"],
    keep=False
)]
print(teacher_conflict_after)

Empty DataFrame
Columns: [full_name, day_of_week, time_slot_id, semester_id, rooms_id, group_id, type]
Index: []


Видаляю дублікати в таблиці й залишаю лише перший запис. Таким чином, конфлікту де вкладачі на одній парі у різних аудиторіях усуваються.

# Два викладача в одній аудиторії

In [ ]:
room_conflicts = df[df.duplicated(subset = ["rooms_id","day_of_week","time_slot_id","semester_id","group_id"],
                                  keep = False)]


print(room_conflicts)

                 full_name  day_of_week  time_slot_id  semester_id  rooms_id  \
457   пан Борислав Олійник            4             5            1        40   
460        Віолетта Швачка            4             5            1        40   
994       Ярослава Каденюк            2             2            3        78   
1000      Амвросій Дашенко            2             2            3        78   
1425        Демʼян Шухевич            4             5            3        45   
1427      Ярослава Каденюк            4             5            3        45   
1592         Богданна Ткач            2             3            4        63   
1595        Пріска Бандера            2             3            4        63   

      group_id          type  
457         59           lab  
460         59           lab  
994          8       seminar  
1000         8       seminar  
1425        60           lab  
1427        60           lab  
1592        20  lecture_hall  
1595        20  lecture_hall  


Два викладача не може одночасно проводити заняття в одній аудиторіЇ в один і той самий день та час в межах одного семестру.

 Бачимо кілька конфліктів:
 * пан Борис Олійник і Віолетта Швачка - в четвер, п'ятою парою, першого семестру. Обидва викладача ведуть пару у 40-ій аудиторії.
 * Ярослава Каденюк і Амвросій Дашенка - у вівторок, другою парою, в третьому семестрі, обидва викладача в 78-ій аудиторії.
 * Дем'ян Шухевич і Ярослава Каденюк - у четвер, п'ятою парою,  третьому сесместрі, обидва викладача стоять у 45-ій аудиторії.
 * Богданна Ткач і Пріска Бандера - у вівторок, третьою парою, у четвертому семестрі, обидва викладача ведуть пару в 63 аудиторії.

In [ ]:
df = df.drop_duplicates(subset=["rooms_id","day_of_week","time_slot_id","semester_id","group_id"],
                         keep = "first")



Видаляю ці дублікати й лишаю лише перший запис, щоб уникнути конфлікту аудиторій.

In [ ]:
room_conflict_after = df[df.duplicated( #Перевіримо чи є ще дублікати після видалення
    subset=["rooms_id", "day_of_week", "time_slot_id", "semester_id","group_id"],
    keep=False
)]
print(room_conflict_after)

Empty DataFrame
Columns: [full_name, day_of_week, time_slot_id, semester_id, rooms_id, group_id, type]
Index: []


# Дві групи в одній аудиторії

In [ ]:
non_lectures = df[~df["type"].isin(["lecture_hall", "seminar"])]
group_conflicts = non_lectures.groupby(
   ["rooms_id", "day_of_week", "time_slot_id", "semester_id","full_name"]
)["group_id"].nunique()
conflicts = group_conflicts[group_conflicts > 1]
if len(conflicts) > 0:
   for idx in conflicts.index:
       print(f"Error: Room {idx[0]} - day {idx[1]} - time {idx[2]} - semester {idx[3]} is double-booked")
else:
   print("Конфліктів аудиторій немає")

Error: Room 75 - day 2 - time 3 - semester 5 is double-booked


In [ ]:
df = df.drop_duplicates(subset = ["rooms_id","day_of_week","time_slot_id","semester_id","full_name"],keep ="first")

In [ ]:
group_conflict_after = df[df.duplicated( #Перевіримо чи є ще дублікати після видалення
    subset=["rooms_id", "day_of_week", "time_slot_id", "semester_id","full_name"],
    keep=False
)]
print(room_conflict_after)

Empty DataFrame
Columns: [full_name, day_of_week, time_slot_id, semester_id, rooms_id, group_id, type]
Index: []


За логікою: лекції та семінари проводяться в декількох групах одночасно в одній аудиторії. А лабораторні роботи не завжди, тому для них перевіряється наявність конфлікту: коли різні викладачі з різними групами займають одну аудиторію в один і той самий час.

# **Висновок:**
Зроблено:
1. Завантажено всі таблиці з бази даних у Google Colab.
 2. З'єднано необхідні таблиці в одну за допомогою pandas.
 3. Виявила конфлікти викладачів - один викладач в один час в двох різних аудиторіях.(Один викладач на дві аудиторії).
 4. Виявила конфлікти аудиторій - одна аудиторія в один час для двох різних викладачів.(Два викладача на одну аудиторію).
 5. Конфлікти усунено методом drop_duplicates із збереженням першого запису.
 6. Для лабораторних робіт  знайдено випадок коли різні групи займають одну аудиторію (Room 75) в один і той самий час в 5 семестрі. Усунула конфлікт шляхом видалення дублікату із збереженням першого запису.
